In [1]:
import os
import pickle
import kagglehub

import numpy as np
import pandas as pd

from typing import List
from scipy.sparse import hstack
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

c:\Users\devas\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data preparation

In [2]:
path = kagglehub.dataset_download("arashnic/book-recommendation-dataset")
books = pd.read_csv(os.path.join(path, 'Books.csv'))
ratings = pd.read_csv(os.path.join(path, 'Ratings.csv'))
users = pd.read_csv(os.path.join(path, 'Users.csv'))

C:\Users\devas\AppData\Local\Temp\ipykernel_6888\1252140016.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv(os.path.join(path, 'Books.csv'))


In [3]:
def get_features(df: pd.DataFrame, cols: List):
    res = df[cols].copy()
    return res

In [4]:
def manage_na(df: pd.DataFrame, col: str):
    df[col] = pd.to_numeric(df[col], errors='coerce').copy()
    df_mean = df['Year-Of-Publication'].mean()
    df[col] = df[col].fillna(df_mean).copy()
    return df

In [5]:
books_df = get_features(books, ['Book-Title', 'Year-Of-Publication', 'Book-Author'])
books_df['Book-Author'] = books_df['Book-Author'].fillna('Unknown Author')
books_df = manage_na(books_df, 'Year-Of-Publication')

users_df = get_features(users, ['Location', 'Age'])

ratings_df = get_features(ratings, ['User-ID', 'Book-Rating'])

model training

In [6]:
scaler = StandardScaler()
tfidf = TfidfVectorizer(stop_words='english')

In [7]:
with open('mern deployment/prediction/model.pkl', 'rb') as file:
    artifacts = pickle.load(file)

nn = artifacts["model"]
features = artifacts["features"]
books_df = artifacts["books_df"]

FileNotFoundError: [Errno 2] No such file or directory: 'mern deployment/prediction/model.pkl'

In [8]:
author_features = tfidf.fit_transform(books_df['Book-Author'])
year_features = scaler.fit_transform(books_df[['Year-Of-Publication']])
features = hstack([author_features, year_features])

qty = 11
nn = NearestNeighbors(n_neighbors=qty, metric='cosine', algorithm='brute')
nn.fit(features)

,n_neighbors,11
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [9]:
def recommend(title: str):
    idx = books_df.index[books_df['Book-Title'] == title][0]
    _, ids = nn.kneighbors(features.tocsr()[idx], n_neighbors=qty)
    recs = books_df.iloc[ids[0][1:]]

    return recs

In [10]:
# title = books_df.iloc[12]['Book-Title']
title = 'Classical Mythology'
recommend(title)

,Book-Title,Year-Of-Publication,Book-Author
95231,Classical Mythology,1998.0,Mark P. O. Morford
193923,Classical mythology,1991.0,Mark P. O Morford
111977,Classical mythology,1985.0,Mark P.O Morford
248513,Ultra 3-D,1994.0,W. Mark
15260,Charity: Stories,1998.0,Mark Richard
1670,Fishboy: A Ghost's Story,1994.0,Mark Richard
36008,The Ice at the Bottom of the World,1991.0,Mark Richard
142601,The Angelspeake Storybook,2000.0,Barbara Mark
93571,The Angelspeake Book Of Prayer And Healing,1997.0,Barbara Mark
29280,Every Mother's Nightmare,1993.0,Mark Thomas


model evaluation

deployment

In [11]:
artifacts = {
    "model": nn,
    "features": features,
    "books_df": books_df,
}

with open("mern deployment/prediction/model.pkl", "wb") as f:
    pickle.dump(artifacts, f)